# 第 53 章：Capstone 开课前预检与学期重启清单

这个 notebook 用一个极小的 reboot-preflight table，对比“只按重启优先级排序”的 naive baseline 和“先检查 blocker、owner、学期刷新与预检状态”的学期重启 workflow。

In [ ]:
from __future__ import annotations

import csv
import platform
from collections import Counter
from pathlib import Path

DATA_PATH = Path('../../data/small/capstone_semester_reboot_preflight_demo.csv')


def load_items(path):
    rows = []
    with path.open(encoding='utf-8', newline='') as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            row['restart_priority_score'] = float(row['restart_priority_score'])
            rows.append(row)
    return rows


def ordered_counts(rows, field):
    counts = Counter(row[field] for row in rows)
    return {key: counts[key] for key in sorted(counts)}


rows = load_items(DATA_PATH)
print(f'Loaded {len(rows)} reboot-preflight items from {DATA_PATH.name}')
print('Python version:', platform.python_version())
print('Startup week:', ordered_counts(rows, 'startup_week'))
print('Preflight status:', ordered_counts(rows, 'preflight_status'))
print('Term refresh:', ordered_counts(rows, 'term_refresh_status'))
print('Owner status:', ordered_counts(rows, 'owner_status'))
print('Blocker status:', ordered_counts(rows, 'blocker_status'))
print('Reference route:', ordered_counts(rows, 'reference_route'))


In [ ]:
reference_ready_ids = sorted(
    row['reboot_id']
    for row in rows
    if row['reference_route'] == 'reboot_ready'
)
top4_priority = sorted(
    rows,
    key=lambda row: row['restart_priority_score'],
    reverse=True,
)[:4]
baseline_ready_hits = sum(
    row['reference_route'] == 'reboot_ready'
    for row in top4_priority
)
baseline_not_ready_hits = sum(
    row['reference_route'] != 'reboot_ready'
    for row in top4_priority
)
baseline_ready_precision = baseline_ready_hits / len(top4_priority)
baseline_not_ready_intrusion = baseline_not_ready_hits / len(top4_priority)

print('Reference reboot-ready items:', reference_ready_ids)
print('Top-4 by restart priority:')
for row in top4_priority:
    print(
        f"  {row['reboot_id']}: priority={row['restart_priority_score']:.2f}, "
        f"route={row['reference_route']}, area={row['reboot_area']}"
    )
print('Baseline reboot precision:', round(baseline_ready_precision, 3))
print('Baseline not-ready intrusion:', round(baseline_not_ready_intrusion, 3))


In [ ]:
def route_reboot_item(row):
    if row['blocker_status'] in {'open', 'high'}:
        return 'resolve_blocker_before_restart', 'course restart still has an open or high blocker'
    if row['owner_status'] != 'assigned':
        return 'assign_startup_owner', 'startup task has no clear owner yet'
    if row['term_refresh_status'] != 'refreshed':
        return 'refresh_term_specific_material', 'dates, links, or term-specific wording still reflect the old semester'
    if row['preflight_status'] != 'complete':
        return 'rerun_preflight_check', 'startup preflight has not been completed yet'
    return 'reboot_ready', 'item has no blocker, has an owner, is refreshed for the new term, and passed preflight'


routed_rows = []
for row in rows:
    route, reason = route_reboot_item(row)
    routed = dict(row)
    routed['route'] = route
    routed['reason'] = reason
    routed_rows.append(routed)

print('Semester reboot workflow routes:')
for row in routed_rows:
    print(
        f"  {row['reboot_id']}: route={row['route']}, "
        f"reference={row['reference_route']}, reason={row['reason']}"
    )


In [ ]:
ready_queue = [row for row in routed_rows if row['route'] == 'reboot_ready']
preflight_queue = [row for row in routed_rows if row['route'] == 'rerun_preflight_check']
refresh_queue = [row for row in routed_rows if row['route'] == 'refresh_term_specific_material']
owner_queue = [row for row in routed_rows if row['route'] == 'assign_startup_owner']
blocker_queue = [row for row in routed_rows if row['route'] == 'resolve_blocker_before_restart']

workflow_accuracy = sum(
    row['route'] == row['reference_route']
    for row in routed_rows
) / len(routed_rows)
ready_precision = sum(
    row['reference_route'] == 'reboot_ready'
    for row in ready_queue
) / max(len(ready_queue), 1)

print('Reboot-ready queue:', [row['reboot_id'] for row in ready_queue])
print('Rerun-preflight queue:', [row['reboot_id'] for row in preflight_queue])
print('Refresh-term queue:', [row['reboot_id'] for row in refresh_queue])
print('Assign-owner queue:', [row['reboot_id'] for row in owner_queue])
print('Resolve-blocker queue:', [row['reboot_id'] for row in blocker_queue])
print('Workflow route accuracy:', round(workflow_accuracy, 3))
print('Structured reboot precision:', round(ready_precision, 3))


In [ ]:
def reboot_steps(row):
    steps = []
    if row['blocker_status'] in {'open', 'high'}:
        steps.append('先解决权限、政策或访问 blocker，并记录解除条件。')
    if row['owner_status'] != 'assigned':
        steps.append('指定新学期启动 owner，并写清负责范围和截止时间。')
    if row['term_refresh_status'] != 'refreshed':
        steps.append('刷新学期特定材料：日期、office hours、入口链接和课程节奏。')
    if row['preflight_status'] != 'complete':
        steps.append('重跑启动前预检：环境、notebook、链接、权限和交付入口。')
    return steps or ['可以进入 semester reboot package；启动时保留检查日期和本学期版本号。']


for row in routed_rows:
    if row['route'] != 'reboot_ready':
        print(f"\n{row['reboot_id']} -> {row['route']} ({row['reboot_area']})")
        for step in reboot_steps(row):
            print(' -', step)


In [ ]:
reboot_outline = [
    'Semester calendar: updated dates, holidays, checkpoints, and week numbering',
    'Entry links: student handout, notebook bundle, TA package, and submission links',
    'Environment and permissions: setup guide, lab machines, private data, and account access',
    'Owner map: who checks, who fixes, who signs off, and who escalates',
    'Policy refresh: AI-use, privacy, grading dates, and public-sharing boundary for this term',
    'Preflight record: who checked what, when, and what changed before launch',
]

reboot_assistant_prompt = '''你是我的 capstone 学期重启助手。

任务：
1. 先阅读 reboot-preflight table，不要只按 restart priority 排序；
2. 先检查 blocker；
3. 再检查 owner、term refresh 和 preflight status；
4. 把每个模块 route 到 reboot_ready、rerun_preflight_check、
   refresh_term_specific_material、assign_startup_owner 或 resolve_blocker_before_restart；
5. 对每个非 ready 模块输出开课前 checklist。

输出格式：
- Reboot-ready modules
- Rerun-preflight modules
- Refresh-term modules
- Assign-owner modules
- Resolve-blocker modules
- Reboot checklist by module
'''

print('Semester reboot outline:')
for item in reboot_outline:
    print(' -', item)

print('\nAssistant prompt:')
print(reboot_assistant_prompt)


In [ ]:
reboot_snapshot = {
    'dataset': DATA_PATH.name,
    'n_reboot_items': len(rows),
    'baseline_top4_reboot_precision': round(baseline_ready_precision, 3),
    'baseline_not_ready_intrusion': round(baseline_not_ready_intrusion, 3),
    'workflow_route_accuracy': round(workflow_accuracy, 3),
    'reboot_ready': [row['reboot_id'] for row in ready_queue],
    'rerun_preflight_check': [row['reboot_id'] for row in preflight_queue],
    'refresh_term_specific_material': [row['reboot_id'] for row in refresh_queue],
    'assign_startup_owner': [row['reboot_id'] for row in owner_queue],
    'resolve_blocker_before_restart': [row['reboot_id'] for row in blocker_queue],
    'python_version': platform.python_version(),
}

print('Semester reboot snapshot:')
for key, value in reboot_snapshot.items():
    print(f'  {key}: {value}')
